In [ ]:
from logging import PlaceHolder

# Step 1: install libraries
!pip install -q chromadb langchain pypdf gradio langchain_community 
!pip install -q google-generativeai langchain_google_genai
!pip install -q sentence-transformers # hugging face as embedding technology

# Step2: install libraries 
import os
from langchain_community.document_loaders import PyPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI 
from langchain_community.embeddings import HuggingFaceEmbeddings 
from langchain_community.vectorstores import Chroma 
from langchain_classic.chains import RetrievalQA
import gradio as gr

# Step 3: load API keys
from google.colab import userdata
os. environ ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Step 4: load and split PDF
pdf_path = "/content/hr_policy.pdf"
loader = PyPDFLoader (pdf_path)
documents = loader.load()

## split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = splitter.split_documents(documents)

# Step 5: Create embeddings and store data in vector database
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

## Store in vector database
vectorstore = Chroma.from_documents(docs, embeddings, collection_name="hr_policy_hf_embeddings")

# Step 6: Using retrieval QA
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)
retriever = vectorstore.as_retriever()
qa_chain = RetrievalQA.from_chain_type(llm, retriever=retriever)

# Step7: Gradio Chatbot
def chatbot(query):
  try:
    response = qa_chain.run(query)
    return response
  except Exception as e:
    return f"Error: {e}"

  
demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(label="Ask your HR assitant a HR policy question", lines=3, placeholder="Type your HR question"),
    outputs=gr.Textbox(label="HR Bot Answer", lines=12),
    title="AI-Powered HR Assistant ChatBot(Gemini + HuggingFace Embeddings)"
)

demo.launch(share=True)

